# Problem Statement

## Customer Churn Prediction

Customer churn is a critical challenge for subscription-based businesses such as telecommunications providers. Acquiring new customers is typically more expensive than retaining existing ones, making churn prediction a valuable tool for improving customer retention strategies.

The goal of this project is to analyze customer behavior and build a predictive model that identifies customers who are likely to churn. By understanding the key factors driving churn, companies can proactively target at-risk customers with retention strategies.

In this project, I use a Telco customer dataset containing demographic information, account details, service subscriptions, and billing data to explore patterns associated with churn and develop a predictive model.

In [67]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

In [68]:
#load data
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [69]:
#Drop Customer ID, doesn't give us extra information
df = df.drop("customerID", axis=1)
#Variables
df.head()
df.shape

(7043, 20)

# Data set 

The dataset contains 7,043 customers and includes information about:
- Customer demographics (e.g., gender, senior citizen status)
- Account information (e.g., tenure, contract type)
- Services subscribed to (e.g., internet service, streaming services)
- Billing details (e.g., monthly charges, payment method)
- Target variable: Churn, indicating whether the customer left the company.
The dataset consists of a mix of:
- Numerical features
- Binary categorical variables
- Multi-category categorical variables

Before analysis, features were converted to appropriate data types to improve analysis and modeling.

In [71]:
#Binary variables 
#find binary features
binary_cols = [col for col in df.columns if df[col].nunique(dropna=True) == 2]
#recode them to 1/0
df[binary_cols] = df[binary_cols].replace({"Yes": 1, "No": 0})


#change to correct data type
num_col = ["tenure", "MonthlyCharges", "TotalCharges"]

for col in num_col:
    df[col] = pd.to_numeric(df[col], errors="coerce")


C:\Users\marle\AppData\Local\Temp\ipykernel_8028\3382558711.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[binary_cols] = df[binary_cols].replace({"Yes": 1, "No": 0})


# Data Cleaning & Preprocessing

To prepare the dataset for analysis, several preprocessing steps were performed:

- Binary categorical variables (e.g., Partner, Dependents, PaperlessBilling) were converted to boolean values.
- Numerical variables were verified and converted to appropriate numeric types.
- Categorical variables with multiple categories were retained as categorical features.

No missing values were identified.

Ensuring correct data types is important for both efficient analysis and accurate modeling.

EDA
- Check for missing value (impute if necessary)
- Check for outliers
- Check for target variable distribution
- Difference of key feature between target 

In [70]:
df.isnull().sum()
#no missing values

gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [72]:
#Check for outliers
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns 
categorical_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns

df.describe()

for col in df[categorical_cols]:
    print(f"\n--- {col} ---")
    print(df[col].value_counts())


--- gender ---
gender
Male      3555
Female    3488
Name: count, dtype: int64

--- MultipleLines ---
MultipleLines
No                  3390
Yes                 2971
No phone service     682
Name: count, dtype: int64

--- InternetService ---
InternetService
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64

--- OnlineSecurity ---
OnlineSecurity
No                     3498
Yes                    2019
No internet service    1526
Name: count, dtype: int64

--- OnlineBackup ---
OnlineBackup
No                     3088
Yes                    2429
No internet service    1526
Name: count, dtype: int64

--- DeviceProtection ---
DeviceProtection
No                     3095
Yes                    2422
No internet service    1526
Name: count, dtype: int64

--- TechSupport ---
TechSupport
No                     3473
Yes                    2044
No internet service    1526
Name: count, dtype: int64

--- StreamingTV ---
StreamingTV
No                     2810
Yes  

# Exploratory Data Analysis (EDA)

Exploratory data analysis is conducted to better understand customer behavior and identify factors associated with churn.

The analysis focuses on:

- Distribution of the target variable
- Relationship between customer characteristics and churn
- Patterns in service usage
- Impact of contract types and billing structures
- Differences in tenure and monthly charges between churned and retained customers

Understanding these relationships helps guide feature engineering and model development.

In [ ]:
#Target variable Churn
df["Churn"].value_counts(normalize=True)

#Imbalance dataset


Churn
0    0.73463
1    0.26537
Name: proportion, dtype: float64

### Class Imbalance

The target variable Churn is imbalanced, with significantly more customers staying than leaving the company.

This imbalance has important implications for modeling:

- Accuracy alone may not be a reliable performance metric.
- A model predicting all customers as non-churners could still achieve relatively high accuracy.

Therefore, additional evaluation metrics such as precision, recall, F1-score, and ROC-AUC will be used.

From a business perspective, identifying customers who are likely to churn is particularly important, making recall for churned customers a key metric.

Imbalanced dataset
- keep in mind for evaluation metrics
- for modeling might be useful to add "class_weight"
- For other project resampling could be useful

In [ ]:
#Feature engineering
